# Classification using confidence

**Difficulty:** Beginner

Classify into a fine-grained category with one `Choice`, then read `confidence` to decide whether to report that specific answer — or, when unsure, the broader category one level up. The hierarchy makes the fallback nearly free: the broad label follows from the narrow one, so there's no second call.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/classification_using_confidence (click to expand)</summary>

# Classification using confidence

> Classify SEC annual reports into 75 industry groups with one Choice each, then read the answer's own confidence to decide whether to report that group or the broader division above it.

Every company filing an annual report with the SEC describes its own business. Classifying under the Standard Industrial Classification (75 industry groups, one `Choice` per document) is mostly easy — a regional bank is a regional bank — but some filings are genuinely hard: a company that just sold one of its two segments, or a startup describing a business it plans to enter rather than one it runs. The model has to pick a group regardless, and a hard answer looks structurally identical to an easy one — until you read `confidence`.

## The recipe: report narrow when sure, broad when not

SIC labels form a hierarchy: 75 industry groups roll up into 10 broader divisions. That makes the fallback nearly free — the broad label follows from the narrow one, no second call needed:

```python
def classify(filing):
    answer = ask(filing["id"], filing["text"])
    sure = answer["confidence"] >= 0.9
    return {
        "level": "group" if sure else "division",
        "label": answer["group"] if sure else division(answer["group"]),
        "confidence": answer["confidence"],
    }
```

The recipe reads `confidence`, not the winning option's own probability — a winner at 0.45 with a close runner-up, and a winner at 0.45 with the rest scattered thinly across many options, are different situations, and `confidence` is what tells them apart.

## Worked results

```
three filings the model was sure about:
  conf 1.00  -> group    28  (chemicals & allied products)
  conf 1.00  -> group    63  (life insurance, accident & health insurance)
  conf 1.00  -> group    49  (electric, gas & sanitary services)

three it was not:
  conf 0.22  -> division manufacturing  (guessed group 38, search/detection/navigation)
  conf 0.23  -> division wholesale trade (guessed group 50, wholesale-durable goods)
  conf 0.29  -> division services        (guessed group 87, engineering/accounting/research)
```

The three confident filings are holding companies on paper, but each names one dominant business outright. The three unsure ones are hard for readable reasons: two are development-stage companies describing a business they *intend* to start, not one they run; the third had two segments and sold one weeks before filing.

## What the broader answer buys, across 60 filings

```
forced to name a group every time      39/60 right
  of the 30 it was sure about          27/30 right  (90%)
  of the 30 it was not                 12/30 right  (40%)

letting it answer coarsely when unsure  48/60 useful answers
```

Where the model was sure, the group it named is right 9 times in 10. Where it wasn't, naming a group anyway was wrong more often than right (40%) — but reporting the *same* answers as a division instead takes that to 70%. Every filing still comes back with a usable label; a hard case gets a coarser one instead of being dropped, guessed, or sent to a second call.

> The full cookbook builds the real two-level SIC taxonomy from the SEC's own published code list (444 codes, 75 groups, 10 divisions) and runs all 60 filings with a results chart — trimmed here to a small 6-group taxonomy. Read the full cookbook at [docs.typesafe.ai/cookbooks/classification_using_confidence](https://docs.typesafe.ai/cookbooks/classification_using_confidence) any time.

</details>

## The core idea

Confidence-gated routing (lesson 7) with an unusually clean fallback: most act/verify/escalate examples need a *different system* for the low-confidence path (a human, a second model). Here, the low-confidence path is just *the same `Choice` answer, read one level coarser* — free, because the hierarchy already encodes it. Whenever your labels form a hierarchy, "fall back to the parent" is worth checking before reaching for a human-review queue or a second model call.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";

// A small 6-group, 3-division taxonomy standing in for the real 75-group SIC hierarchy.
const GROUPS: Record<string, { title: string; division: string }> = {
  "28": { title: "Chemicals & Allied Products", division: "Manufacturing" },
  "35": { title: "Industrial Machinery & Equipment", division: "Manufacturing" },
  "60": { title: "Depository Institutions", division: "Finance" },
  "63": { title: "Insurance", division: "Finance" },
  "73": { title: "Business Services", division: "Services" },
  "87": { title: "Engineering & Management Services", division: "Services" },
};

const CONFIDENT_THRESHOLD = 0.9;

async function classify(text: string) {
  const criteria = Object.fromEntries(
    Object.entries(GROUPS).map(([code, { title }]) => [code, title]),
  );
  const response = await client.systemOne({
    state: text,
    questions: {
      group: choice(
        "Which broad industry does this company operate in? Judge the company's own operations as this filing describes them.",
        criteria,
      ),
    },
  });
  const { choice: group, confidence } = response.answers.group;
  const sure = confidence >= CONFIDENT_THRESHOLD;
  return {
    level: sure ? "group" : "division",
    label: sure ? group : GROUPS[group].division,
    confidence,
    group,
  };
}

// One easy filing, one genuinely ambiguous one (a development-stage shell company,
// the same shape as two of the real cookbook's hardest cases).
const filings: Record<string, string> = {
  confident:
    "We are a chemical manufacturer producing industrial coatings and specialty polymers for the automotive and construction industries.",
  ambiguous:
    "The Company was organized to enter into the computer security software industry. We have not yet commenced principal operations and have generated no revenue to date.",
};

for (const [name, text] of Object.entries(filings)) {
  const result = await classify(text);
  console.log(name, "->", result);
}

## What just happened

`confident` names a specific chemical-manufacturing business with no ambiguity — expect `level: "group"`, code `28`, confidence at or near `1.0`. `ambiguous` is a shell company that hasn't started operating yet and only names an *intended* industry, the same shape as the real cookbook's development-stage cases — expect confidence well under `0.9`, falling back to a division rather than committing to a specific, possibly-wrong group.

**Try it yourself:** rewrite `ambiguous` to still be a pre-revenue company, but have it commit unambiguously to one specific business ("we have signed our first engineering services contract and expect to recognize revenue next quarter"). Predict whether that's enough to push confidence back over the `0.9` line despite still being early-stage.

## Related

- `lessons/patterns/0007-confidence-gated-routing.ipynb` — the general act/verify/escalate pattern this cookbook specializes into "report narrow or report broad"
- `cookbooks/classification/hierarchical-classification.ipynb` — another cookbook that exploits a label hierarchy, there to search deeper instead of to fall back shallower

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.